# Project Directory

In [38]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


# Load VADB Dataset

In [39]:
# CELL 3 — LOAD VADB DATASET

from datasets import load_dataset
import pandas as pd

# Load dataset
ds = load_dataset(
    "BestiVictoryLab/VADB",
    data_files="train.csv"
)

# Convert to DataFrame
df = ds["train"].to_pandas()

# Display dataset summary
print("\n[LOAD STATUS] SUCCESS")
print(f"[DATASET]      BestiVictoryLab/VADB")
print(f"[SPLIT]        Train")
print(f"[RECORDS]      {len(df):,}")
print(f"[COLUMNS]      {len(df.columns)}")
print(f"[SHAPE]        {df.shape}")

print("\n[AVAILABLE COLUMNS]")
for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")




[LOAD STATUS] SUCCESS
[DATASET]      BestiVictoryLab/VADB
[SPLIT]        Train
[RECORDS]      10,488
[COLUMNS]      17
[SHAPE]        (10488, 17)

[AVAILABLE COLUMNS]
01. video_id
02. score
03. composition
04. shotsize
05. lighting
06. visualtone
07. color
08. depthoffield
09. expression
10. movement
11. costume
12. makeup
13. label
14. Unnamed: 13
15. Unnamed: 14
16. Unnamed: 15
17. Unnamed: 16


In [40]:
# Save VADB source dataset locally
source_path = RAW_DIR / "vadb_train.csv"
df.to_csv(source_path, index=False)

print(f"File: {source_path}")
print(f"Records: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Size: {source_path.stat().st_size:,} bytes")
print("[STATUS] SUCCESS")
print("=" * 55)

File: /content/week3_project/data/raw/vadb_train.csv
Records: 10,488
Columns: 17
Size: 1,300,270 bytes
[STATUS] SUCCESS


# Source Selection and Lineage Utilities

In [41]:
 #Cell 4 — REQUIRED: Source Selection + Lineage Utilities

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]
def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()
def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


# Common Lineage Registry

In [42]:
# Cell 4A — Common lineage registry

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source)
    output = Path(output)

    if not source.exists():
        raise FileNotFoundError(f"Source does not exist: {source}")

    if not output.exists():
        raise FileNotFoundError(f"Output does not exist: {output}")

    if output.stat().st_size == 0:
        raise ValueError(f"Output is empty: {output}")

    if not processing or not isinstance(processing, str):
        raise ValueError("Processing description must be provided.")

    if RAW_DIR not in source.parents:
        raise ValueError(f"Source must be under RAW_DIR: {RAW_DIR}")

    if PROCESSED_DIR not in output.parents:
        raise ValueError(f"Output must be under PROCESSED_DIR: {PROCESSED_DIR}")

    lineage_entries[str(output)] = {
    "source": source,
    "output": output,
    "processing": processing
}

    print("Registered:", source.name, "->", output.name)


print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


In [43]:
# VADB dataset inspection

df = ds["train"].to_pandas()

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (10488, 17)

Columns:
['video_id', 'score', 'composition', 'shotsize', 'lighting', 'visualtone', 'color', 'depthoffield', 'expression', 'movement', 'costume', 'makeup', 'label', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']

Missing values:
video_id            0
score               0
composition         0
shotsize            0
lighting            0
visualtone          0
color               0
depthoffield        0
expression       2355
movement         2355
costume          2355
makeup           2355
label               0
Unnamed: 13     10488
Unnamed: 14     10488
Unnamed: 15     10488
Unnamed: 16     10487
dtype: int64

Duplicate rows: 0

First 5 rows:


,video_id,score,composition,shotsize,lighting,visualtone,color,depthoffield,expression,movement,costume,makeup,label,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,2VOORIZOoHUAAAAAAAAAAAAAK8vRAQBr,3.600000,3.900000,4.100000,3.600000,3.600000,3.900000,3.800000,3.800000,4.100000,4.200000,3.800000,Characters (Film and Television),NaN,NaN,NaN,None
1,70pYTobVJbAAAAAAAAAAAAAAK8vRAQBr,3.900000,3.800000,3.700000,4.100000,4.000000,4.222222,3.700000,4.200000,4.100000,4.000000,4.200000,Characters (Film and Television),NaN,NaN,NaN,None
2,A*--twRqwniGYAAAAAAAAAAAAAcKh3AA,4.250000,4.250000,4.000000,4.375000,4.375000,4.125000,4.125000,4.750000,4.375000,4.625000,4.500000,Characters (Film and Television),NaN,NaN,NaN,None
3,A*-3CPQaetYzAAAAAAAAAAAAAADtp1AQ,3.300000,3.300000,3.300000,3.300000,3.300000,3.600000,3.200000,3.300000,3.400000,3.700000,3.300000,Characters (Film and Television),NaN,NaN,NaN,None
4,A*-6YVS5aetpgAAAAAAAAAAAAAcKh3AA,3.444444,3.555556,3.444444,3.555556,3.333333,3.333333,3.111111,3.222222,3.555556,3.888889,2.111111,Characters (Film and Television),NaN,NaN,NaN,None


# 	Tabular Data Preparation

In [44]:
# Cell 5B — PATHWAY B: VADB CSV preparation

import pandas as pd

# Step 1: Save a local copy of the downloaded source dataset
source_path = RAW_DIR / "vadb_train.csv"
source_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(source_path, index=False)

# Step 2: Select and load the exact source file
f = select_source(relative_path="vadb_train.csv")

df = pd.read_csv(f)

# Step 3: Inspect original data
print("--- Original Dataset ---")
print("Source:", f.name)
print("Rows/columns:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head().to_string(index=False))
print("Missing per column:", df.isna().sum().to_dict())
print("Exact duplicate rows:", int(df.duplicated().sum()))

assert not df.empty, "The selected dataset is empty."

# Step 4: Preparation settings
# Preserve original data until cleaning is justified.
REMOVE_DUPLICATES = False
TRIM_TEXT = False

prepared = df.copy()

if REMOVE_DUPLICATES:
    prepared = prepared.drop_duplicates().copy()

if TRIM_TEXT:
    for col in prepared.select_dtypes(
        include=["object", "string"]
    ).columns:
        prepared[col] = prepared[col].astype("string").str.strip()

# Step 5: Compare before and after
print("\n--- Preparation Comparison ---")
print("Original rows:", len(df))
print("Prepared rows:", len(prepared))
print("Rows removed:", len(df) - len(prepared))
print("Missing values after:", prepared.isna().sum().to_dict())

# Step 6: Export prepared data
prepared_path = PROCESSED_DIR / "prepared_table.csv"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
prepared.to_csv(prepared_path, index=False)

# Step 7: Validate exported data
assert prepared_path.is_file()
assert prepared_path.stat().st_size > 0
assert not prepared.empty

exported = pd.read_csv(prepared_path)

assert len(exported) == len(prepared)
assert exported.columns.tolist() == prepared.columns.tolist()

# Step 8: Display evidence
print("\nPATHWAY B completed successfully.")
print("Remove duplicates:", REMOVE_DUPLICATES)
print("Trim text:", TRIM_TEXT)
print("Prepared shape:", prepared.shape)
print("Saved:", prepared_path)
print("Validation: PASS")

--- Original Dataset ---
Source: vadb_train.csv
Rows/columns: (10488, 17)
Columns: ['video_id', 'score', 'composition', 'shotsize', 'lighting', 'visualtone', 'color', 'depthoffield', 'expression', 'movement', 'costume', 'makeup', 'label', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16']
                        video_id    score  composition  shotsize  lighting  visualtone    color  depthoffield  expression  movement  costume   makeup                            label  Unnamed: 13  Unnamed: 14  Unnamed: 15 Unnamed: 16
2VOORIZOoHUAAAAAAAAAAAAAK8vRAQBr 3.600000     3.900000  4.100000  3.600000    3.600000 3.900000      3.800000    3.800000  4.100000 4.200000 3.800000 Characters (Film and Television)          NaN          NaN          NaN         NaN
70pYTobVJbAAAAAAAAAAAAAAK8vRAQBr 3.900000     3.800000  3.700000  4.100000    4.000000 4.222222      3.700000    4.200000  4.100000 4.000000 4.200000 Characters (Film and Television)          NaN          NaN          NaN         NaN

In [45]:
# Inspect suspicious unnamed columns

unnamed_cols = [col for col in df.columns if col.startswith("Unnamed")]

for col in unnamed_cols:
    print(f"\n{col}")
    print("Non-missing values:", df[col].dropna().unique()[:10])
    print("Non-missing count:", df[col].notna().sum())


Unnamed: 13
Non-missing values: []
Non-missing count: 0

Unnamed: 14
Non-missing values: []
Non-missing count: 0

Unnamed: 15
Non-missing values: []
Non-missing count: 0

Unnamed: 16
Non-missing values: ['3.7575176,3.6068854,3.733898,3.7239804,3.787685,3.6134713,3.9,4.1,3.6,3.6,3.9,3.8']
Non-missing count: 1


In [46]:
# Find the row containing the unexpected value

affected_row = df[df["Unnamed: 16"].notna()]

print("Affected row:")
display(affected_row.T)

print("\nRow index:", affected_row.index.tolist())
print("\nUnexpected value:")
print(affected_row["Unnamed: 16"].iloc[0])

Affected row:


,6
video_id,A*-8KCT5GfJC8AAAAAAAAAAAAAcKh3AA
score,3.357143
composition,3.214286
shotsize,3.142857
lighting,3.428571
visualtone,3.214286
color,3.357143
depthoffield,3.5
expression,3.142857
movement,3.357143



Row index: [6]

Unexpected value:
3.7575176,3.6068854,3.733898,3.7239804,3.787685,3.6134713,3.9,4.1,3.6,3.6,3.9,3.8


# Preparation Approach Comparison and Validation

In [47]:
# B3 — Compare two preparation approaches

# Approach 1: Preserve all original columns
approach_1 = df.copy()

# Approach 2: Remove unintended Unnamed columns
expected_columns = [
    "video_id", "score", "composition", "shotsize",
    "lighting", "visualtone", "color", "depthoffield",
    "expression", "movement", "costume", "makeup", "label"
]

approach_2 = df[expected_columns].copy()

# Compare both approaches
print("--- B3 Preparation Comparison ---")

print("\nApproach 1: Original structure")
print("Shape:", approach_1.shape)
print("Columns:", len(approach_1.columns))

print("\nApproach 2: Remove unintended columns")
print("Shape:", approach_2.shape)
print("Columns:", len(approach_2.columns))

print("\nRecords preserved:", len(approach_2) == len(df))
print("Missing values preserved:")
print(approach_2.isna().sum().to_dict())

# Validate selected output
assert approach_2.shape == (10488, 13)
assert approach_2["video_id"].notna().all()
assert approach_2["label"].notna().all()

prepared = approach_2.copy()

prepared_path = PROCESSED_DIR / "prepared_table.csv"
prepared.to_csv(prepared_path, index=False)

exported = pd.read_csv(prepared_path)

assert exported.shape == approach_2.shape
assert exported.columns.tolist() == expected_columns

print("\nSelected approach: Approach 2")
print("Reason: Removes unintended columns while preserving all records.")
print("Prepared output:", prepared_path)
print("B3 Validation: PASS")

--- B3 Preparation Comparison ---

Approach 1: Original structure
Shape: (10488, 17)
Columns: 17

Approach 2: Remove unintended columns
Shape: (10488, 13)
Columns: 13

Records preserved: True
Missing values preserved:
{'video_id': 0, 'score': 0, 'composition': 0, 'shotsize': 0, 'lighting': 0, 'visualtone': 0, 'color': 0, 'depthoffield': 0, 'expression': 2355, 'movement': 2355, 'costume': 2355, 'makeup': 2355, 'label': 0}

Selected approach: Approach 2
Reason: Removes unintended columns while preserving all records.
Prepared output: /content/week3_project/data/processed/prepared_table.csv
B3 Validation: PASS


# Register Data Lineage

In [48]:
# B4.1 — Register Pathway B lineage

register_lineage(
    source=f,
    output=prepared_path,
    processing=(
        "CSV preparation: removed four unintended Unnamed columns; "
        "preserved all 10,488 records and original missing values; "
        "no duplicate removal or text trimming."
    )
)

print("\nLineage registration completed.")
print("Registered entries:", len(lineage_entries))

Registered: vadb_train.csv -> prepared_table.csv

Lineage registration completed.
Registered entries: 1


# Generate Common Lineage Manifest

In [49]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):

    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 1
Saved to: /content/week3_project/data/week3_manifest.json


#	Validate Common Lineage Manifest

In [50]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 1
Manifest: /content/week3_project/data/week3_manifest.json


#	Integrated V1 Pipeline Utilities

In [51]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


# VADB Tabular Pipeline Execution

In [52]:
# Cell 8B — VADB CSV Pipeline
import pandas as pd

# Step 1: Select source and configuration
SOURCE_B = resolve_source("vadb_train.csv")

CONFIG_B = {
    "remove_exact_duplicates": False,
    "trim_text": False,
    "keep_columns": [
        "video_id", "score", "composition", "shotsize",
        "lighting", "visualtone", "color", "depthoffield",
        "expression", "movement", "costume", "makeup", "label"
    ]
}

# Step 2: Define reusable table loader
def read_table(src):
    if src.suffix.lower() == ".csv":
        return pd.read_csv(src)
    elif src.suffix.lower() == ".xlsx":
        return pd.read_excel(src)
    else:
        raise ValueError("Expected a CSV or XLSX source")


# Step 3: Inspect source dataset
def inspect_table(inputs, cfg):
    df = read_table(inputs[0])

    if df.empty:
        raise ValueError("Empty input table")

    return {
        "rows": len(df),
        "columns": df.columns.tolist(),
        "missing": {
            str(k): int(v)
            for k, v in df.isna().sum().items()
        },
        "exact_duplicates": int(df.duplicated().sum())
    }


# Step 4: Prepare tabular dataset
def prepare_table(inputs, out, cfg):
    src = inputs[0]
    df = read_table(src)

    result = df.copy()

    # Retain only the 13 expected VADB columns
    missing_columns = [
        col for col in cfg["keep_columns"]
        if col not in result.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Required columns missing: {missing_columns}"
        )

    result = result[cfg["keep_columns"]].copy()

    # Apply Part B preparation decisions
    if cfg["remove_exact_duplicates"]:
        result = result.drop_duplicates().copy()

    if cfg["trim_text"]:
        for col in result.select_dtypes(
            include=["object", "string"]
        ).columns:
            result[col] = (
                result[col].astype("string").str.strip()
            )

    if result.empty:
        raise ValueError("No prepared table rows")

    # Step 5: Save prepared data
    target = out / "prepared_table.csv"
    result.to_csv(target, index=False)

    # Step 6: Describe actual processing
    method = (
        "Retained 13 expected VADB columns; "
        "excluded four unintended Unnamed columns; "
        f"remove_exact_duplicates={cfg['remove_exact_duplicates']}; "
        f"trim_text={cfg['trim_text']}; "
        "preserved missing values."
    )

    # Step 7: Source-to-output mapping
    mapping = [
        {
            "source": str(src),
            "output": str(target)
        }
    ]

    return [target], mapping, method


# Step 8: Execute and register pipeline
run_B = register_and_run(
    "tabular",
    SOURCE_B,
    inspect_table,
    prepare_table,
    CONFIG_B
)

print("Cell 8B completed successfully.")


SUCCESS: tabular | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/tabular/pipeline_manifest.json
Cell 8B completed successfully.


# Repeatability and Provenance Verification

In [53]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )
    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )
    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: tabular | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/tabular/pipeline_manifest.json
PASS: tabular — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


In [54]:
prepared_path = PIPE_DIR / "tabular" / "prepared_table.csv"

check_df = pd.read_csv(prepared_path)

print("Prepared dataset shape:", check_df.shape)
print("\nColumns:")
print(check_df.columns.tolist())

print("\nFirst 3 records:")
display(check_df.head(3))

print("\nFile size:", prepared_path.stat().st_size, "bytes")

Prepared dataset shape: (10488, 13)

Columns:
['video_id', 'score', 'composition', 'shotsize', 'lighting', 'visualtone', 'color', 'depthoffield', 'expression', 'movement', 'costume', 'makeup', 'label']

First 3 records:


,video_id,score,composition,shotsize,lighting,visualtone,color,depthoffield,expression,movement,costume,makeup,label
0,2VOORIZOoHUAAAAAAAAAAAAAK8vRAQBr,3.60,3.90,4.1,3.600,3.600,3.900000,3.800,3.80,4.100,4.200,3.8,Characters (Film and Television)
1,70pYTobVJbAAAAAAAAAAAAAAK8vRAQBr,3.90,3.80,3.7,4.100,4.000,4.222222,3.700,4.20,4.100,4.000,4.2,Characters (Film and Television)
2,A*--twRqwniGYAAAAAAAAAAAAAcKh3AA,4.25,4.25,4.0,4.375,4.375,4.125000,4.125,4.75,4.375,4.625,4.5,Characters (Film and Television)



File size: 1258187 bytes
